In [1]:
import os
from dotenv import load_dotenv

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print('No API key was found!')
elif not api_key.startswith('sk-proj-'):
    print('An API key was found, but it doesnt start with "sk-proj-", i.e, it is not an OpenAI API key')
elif api_key.strip() != api_key:
    print('An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them.')
else:
    print('The API key found and looks good so far!')


The API key found and looks good so far!


# LangChain

In [3]:
import os
import sys
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# --- Configuration ---
# Ensure your API key environment variable is set (e.g., OPENAI_API_KEY)
try:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
    print(f"Language model initialized: {llm.model_name}")
except Exception as e:
    print(f"Error initializing language model: {e}", file=sys.stderr)
    print("Please ensure your OPENAI_API_KEY is set correctly.", file=sys.stderr)
    sys.exit(1) # Exit if the LLM cannot be initialized


# --- Define Chain Components ---

# 1. Initial Generation: Creates the first draft of the product description.
# The input to this chain will be a dictionary, so we update the prompt template.
generation_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Write a short, simple product description for a new smart coffee mug."),
        ("user", "{product_details}")
    ])
    | llm
    | StrOutputParser()
)

# 2. Critique: Evaluates the generated description and provides feedback.
critique_chain = (
    ChatPromptTemplate.from_messages([
        ("system", """Critique the following product description based on clarity, conciseness, and appeal.
        Provide specific suggestions for improvement."""),
        # This will receive 'initial_description' from the previous step.
        ("user", "Product Description to Critique:\n{initial_description}")
    ])
    | llm
    | StrOutputParser()
)

# 3. Refinement: Rewrites the description based on the original details and the critique.
refinement_chain = (
    ChatPromptTemplate.from_messages([
        ("system", """Based on the original product details and the following critique,
        rewrite the product description to be more effective.

        Original Product Details: {product_details}
        Critique: {critique}

        Refined Product Description:"""),
        ("user", "") # User input is empty as the context is provided in the system message
    ])
    | llm
    | StrOutputParser()
)


# --- Build the Full Reflection Chain (Refactored) ---
# This chain is structured to be more readable and linear.
full_reflection_chain = (
    RunnablePassthrough.assign(
        initial_description=generation_chain
    )
    | RunnablePassthrough.assign(
        critique=critique_chain
    )
    | refinement_chain
)


# --- Run the Chain ---
async def run_reflection_example(product_details: str):
    """Runs the LangChain reflection example with product details."""
    print(f"\n--- Running Reflection Example for Product: '{product_details}' ---")
    try:
        # The chain now expects a dictionary as input from the start.
        final_refined_description = await full_reflection_chain.ainvoke(
            {"product_details": product_details}
        )
        print("\n--- Final Refined Product Description ---")
        print(final_refined_description)
    except Exception as e:
        print(f"\nAn error occurred during chain execution: {e}")

test_product_details = "A mug that keeps coffee hot and can be controlled by a smartphone app."
# The notebook already runs an event loop, so await the coroutine directly.
await run_reflection_example(test_product_details)

Language model initialized: gpt-4o-mini

--- Running Reflection Example for Product: 'A mug that keeps coffee hot and can be controlled by a smartphone app.' ---

--- Final Refined Product Description ---
Introducing the Smart Coffee Mug – the ultimate companion for coffee lovers who crave perfection! This innovative mug ensures your coffee stays at the ideal temperature for hours, so every sip is as enjoyable as the first. With an intuitive smartphone app compatible with both iOS and Android, you can effortlessly customize the temperature to your liking and receive notifications when your drink reaches the perfect heat.

Crafted from high-quality stainless steel for durability and long-lasting use, the Smart Coffee Mug is designed for those on the go. Picture yourself cradling this sleek mug in your hands, experiencing the comforting warmth as you savor each delicious sip of your perfectly heated brew.

Don't settle for lukewarm coffee – elevate your coffee experience today with the S

# Google ADK

In [4]:
from google.adk.agents import SequentialAgent, LlmAgent
from google.adk.runners import InMemoryRunner
from google.genai import types

GEMINI_MODEL = "gemini-flash-latest"

# The first agent generates the initial draft.
generator = LlmAgent(
    name="DraftWriter",
    model=GEMINI_MODEL,
    description="Generates initial draft content on a given subject.",
    instruction="Write a short, informative paragraph about the user's subject.",
    output_key="draft_text" # The output is saved to this state key.
)

# The second agent critiques the draft from the first agent.
reviewer = LlmAgent(
    name="FactChecker",
    model=GEMINI_MODEL,
    description="Reviews a given text for factual accuracy and provides a structured critique.",
    instruction="""
    You are a meticulous fact-checker.
    1. Read the text provided in the state key 'draft_text':
       {draft_text}
    2. Carefully verify the factual accuracy of all claims.
    3. Your final output must be a dictionary containing two keys:
       - "status": A string, either "ACCURATE" or "INACCURATE".
       - "reasoning": A string providing a clear explanation for your status, citing specific issues if any are found.
    """,
    output_key="review_output" # The structured dictionary is saved here.
)

# The SequentialAgent ensures the generator runs before the reviewer.
review_pipeline = SequentialAgent(
    name="WriteAndReview_Pipeline",
    sub_agents=[generator, reviewer]
)

# Execution Flow:
# 1. generator runs -> saves its paragraph to state['draft_text'].
# 2. reviewer runs -> reads state['draft_text'] and saves its dictionary output to state['review_output'].

# --- Run the Pipeline ---
APP_NAME = "write_and_review_app"
USER_ID = "user_1"

runner = InMemoryRunner(agent=review_pipeline, app_name=APP_NAME)
session = await runner.session_service.create_session(app_name=APP_NAME, user_id=USER_ID)

message = types.Content(
    role="user",
    parts=[types.Part(text="The history of the Apollo space program.")],
)

async for event in runner.run_async(
    user_id=USER_ID, session_id=session.id, new_message=message
):
    if event.is_final_response() and event.content and event.content.parts:
        text = event.content.parts[0].text
        if text:
            print(f"--- {event.author} ---")
            print(text)
            print()


/var/folders/_n/81b2c8ld4ybgj_p8m3_g6rc00000gn/T/ipykernel_2535/117378132.py:34: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  review_pipeline = SequentialAgent(
Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


--- DraftWriter ---
Initiated by NASA in 1961 following President John F. Kennedy’s challenge to land an American on the Moon before the end of the decade, the Apollo space program stands as one of humanity's greatest technological and exploratory achievements. Despite early adversity, including the tragic Apollo 1 launchpad fire in 1967, the program advanced rapidly, culminating in the historic Apollo 11 mission on July 20, 1969, when Neil Armstrong and Buzz Aldrin became the first humans to walk on the lunar surface. Running through Apollo 17 in December 1972, the program successfully landed six crews—a total of twelve astronauts—on the Moon, returning invaluable scientific data and hundreds of pounds of lunar samples while decisively securing American leadership in the Cold War Space Race.

--- FactChecker ---
```json
{
  "status": "ACCURATE",
  "reasoning": "All statements in the draft text are factually accurate. President John F. Kennedy challenged the nation in 1961 to land a ma

In [5]:
# Same write-and-review pipeline, written with the graph-based Workflow API (google-adk 2.x).
# SequentialAgent is deprecated in favor of Workflow.
# Requires GOOGLE_API_KEY in .env (loaded above).

from google.adk import Agent, Workflow
from google.adk.runners import InMemoryRunner
from google.genai import types

GEMINI_MODEL = "gemini-flash-latest"

# The first node generates the initial draft.
generator = Agent(
    name="DraftWriter",
    model=GEMINI_MODEL,
    description="Generates initial draft content on a given subject.",
    instruction="Write a short, informative paragraph about the user's subject.",
)

# The second node critiques the draft it receives from the first node.
reviewer = Agent(
    name="FactChecker",
    model=GEMINI_MODEL,
    description="Reviews a given text for factual accuracy and provides a structured critique.",
    instruction="""
    You are a meticulous fact-checker.
    1. Read the draft paragraph you receive as input.
    2. Carefully verify the factual accuracy of all claims.
    3. Your final output must be a dictionary containing two keys:
       - "status": A string, either "ACCURATE" or "INACCURATE".
       - "reasoning": A string providing a clear explanation for your status, citing specific issues if any are found.
    """,
)

# The edge from generator to reviewer both orders the two nodes and hands the
# draft to the reviewer as its input, so no state key is needed.
review_pipeline = Workflow(
    name="WriteAndReview_Pipeline",
    edges=[("START", generator, reviewer)],
)

# --- Run the Pipeline ---
APP_NAME = "write_and_review_workflow_app"
USER_ID = "user_1"

runner = InMemoryRunner(agent=review_pipeline, app_name=APP_NAME)
session = await runner.session_service.create_session(app_name=APP_NAME, user_id=USER_ID)

message = types.Content(
    role="user",
    parts=[types.Part(text="The history of the Apollo space program.")],
)

async for event in runner.run_async(
    user_id=USER_ID, session_id=session.id, new_message=message
):
    if event.is_final_response() and event.content and event.content.parts:
        text = event.content.parts[0].text
        if text:
            print(f"--- {event.author} ---")
            print(text)
            print()


--- DraftWriter ---
Initiated by NASA in 1961 and galvanized by President John F. Kennedy’s ambition to land a human on the Moon before the decade's end, the Apollo space program stands as one of humanity's greatest technological achievements. Despite a devastating setback in 1967 with the fatal Apollo 1 launchpad fire, the program rapidly advanced, culminating on July 20, 1969, when Apollo 11 astronauts Neil Armstrong and Buzz Aldrin successfully set foot on the lunar surface. Between 1969 and the program's conclusion in 1972 with Apollo 17, a total of 12 American astronauts walked on the Moon, conducting scientific research and collecting hundreds of pounds of lunar samples. Beyond winning the Cold War Space Race for the United States, the Apollo program catalyzed groundbreaking advancements in computing, aerospace engineering, and telecommunications that transformed modern technology.

--- FactChecker ---
```json
{
  "status": "ACCURATE",
  "reasoning": "All statements in the draft 

## Interactive Loop

In [6]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage

# --- Configuration ---
# Load environment variables from .env file (for OPENAI_API_KEY)
load_dotenv()

# Check if the API key is set
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in .env file. Please add it.")

# Initialize the Chat LLM. We use a powerful model like gpt-4o for better reasoning.
# A lower temperature is used for more deterministic and focused outputs.
llm = ChatOpenAI(model="gpt-4o", temperature=0.1)


def run_reflection_loop():
    """
    Demonstrates a multi-step AI reflection loop to progressively improve a Python function.
    """

    # --- The Core Task ---
    task_prompt = """
    Your task is to create a Python function named `calculate_factorial`.
    This function should do the following:
    1.  Accept a single integer `n` as input.
    2.  Calculate its factorial (n!).
    3.  Include a clear docstring explaining what the function does.
    4.  Handle edge cases: The factorial of 0 is 1.
    5.  Handle invalid input: Raise a ValueError if the input is a negative number.
    """

    # --- The Reflection Loop ---
    max_iterations = 3
    current_code = ""
    # We will build a conversation history to provide context in each step.
    message_history = [HumanMessage(content=task_prompt)]

    for i in range(max_iterations):
        print("\n" + "="*25 + f" REFLECTION LOOP: ITERATION {i + 1} " + "="*25)

        # --- 1. GENERATE / REFINE STAGE ---
        # In the first iteration, it generates. In subsequent iterations, it refines.
        if i == 0:
            print("\n>>> STAGE 1: GENERATING initial code...")
            # The first message is just the task prompt.
            response = llm.invoke(message_history)
            current_code = response.content
        else:
            print("\n>>> STAGE 1: REFINING code based on previous critique...")
            # The message history now contains the task, the last code, and the last critique.
            # We instruct the model to apply the critiques.
            message_history.append(HumanMessage(content="Please refine the code using the critiques provided."))
            response = llm.invoke(message_history)
            current_code = response.content

        print("\n--- Generated Code (v" + str(i + 1) + ") ---\n" + current_code)
        message_history.append(response) # Add the generated code to history

        # --- 2. REFLECT STAGE ---
        print("\n>>> STAGE 2: REFLECTING on the generated code...")

        # Create a specific prompt for the reflector agent.
        # This asks the model to act as a senior code reviewer.
        reflector_prompt = [
            SystemMessage(content="""
                You are a senior software engineer and an expert in Python.
                Your role is to perform a meticulous code review.
                Critically evaluate the provided Python code based on the original task requirements.
                Look for bugs, style issues, missing edge cases, and areas for improvement.
                If the code is perfect and meets all requirements, respond with the single phrase 'CODE_IS_PERFECT'.
                Otherwise, provide a bulleted list of your critiques.
            """),
            HumanMessage(content=f"Original Task:\n{task_prompt}\n\nCode to Review:\n{current_code}")
        ]

        critique_response = llm.invoke(reflector_prompt)
        critique = critique_response.content

        # --- 3. STOPPING CONDITION ---
        if "CODE_IS_PERFECT" in critique:
            print("\n--- Critique ---\nNo further critiques found. The code is satisfactory.")
            break

        print("\n--- Critique ---\n" + critique)
        # Add the critique to the history for the next refinement loop.
        message_history.append(HumanMessage(content=f"Critique of the previous code:\n{critique}"))

    print("\n" + "="*30 + " FINAL RESULT " + "="*30)
    print("\nFinal refined code after the reflection process:\n")
    print(current_code)


if __name__ == "__main__":
    run_reflection_loop()



========================= REFLECTION LOOP: ITERATION 1 =========================

>>> STAGE 1: GENERATING initial code...

--- Generated Code (v1) ---
Here's a Python function named `calculate_factorial` that meets the specified requirements:

```python
def calculate_factorial(n):
    """
    Calculate the factorial of a non-negative integer n.

    The factorial of a non-negative integer n is the product of all positive integers less than or equal to n.
    The factorial of 0 is defined as 1.

    Parameters:
    n (int): A non-negative integer whose factorial is to be calculated.

    Returns:
    int: The factorial of the input integer n.

    Raises:
    ValueError: If the input is a negative integer.
    """
    if not isinstance(n, int):
        raise ValueError("Input must be an integer.")
    if n < 0:
        raise ValueError("Factorial is not defined for negative numbers.")
    
    if n == 0:
        return 1
    
    factorial = 1
    for i in range(1, n + 1):
        fact